#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim,col

In [0]:
rename_map = {
    'CID':'customer_number',
    'BDATE':'birthdate',
    'GEN':'gender'}

#Reading Data From Bronze Layer

In [0]:
df = spark.table('workspace.bronze.erp_cust_az12')
df.display()

#Data Cleaning

##Trimming String columns

In [0]:
for column in df.dtypes:
    if column[1] == 'string':
        df = df.withColumn(column[0],trim(col(column[0])))

##Substring extraction

In [0]:
df = (
    df.withColumn(
        'cid',
        F.when (F.upper(df['cid']).startswith('NAS'),F.substring(df['cid'],4,F.length(df['cid']))).otherwise(df['cid']))

    )

##Handling Invalid Values

In [0]:
df = df.withColumn('bdate',F.when (df['bdate'] > F.curdate(), None ).otherwise(df['bdate']))

##Standardization

###Value Mapping (e.g., M → Male)


In [0]:
df =(
      df.withColumn(
          'GEN',
            F.when (F.upper(df['GEN']).isin ('M','MALE'),'Male')
            .when (F.upper(df['GEN']).isin ('F','FEMALE'),'Female')
            .otherwise('Unknown')
            )
    )


###Column Naming Standardization

In [0]:
for old_name,new_name in rename_map.items():
    df = df.withColumnRenamed(old_name,new_name)

#Loading Data Into Silver Layer

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('silver.erp_customer_az12')